# NME 2: Visualizing Data

During the last NME, we went over how to obtain/import data using CSVs and how to leverage the pandas and NumPy Python libraries in order to explore datasets.

In a typical data science workflow, the next step will be to visualize the data. Visualizing data is important because it helps us identify patterns and insights in our data. Today, we'll go over how to use the Matplotlib and Seaborn libraries to create data visualizations such as plot and tables.

> **Goal for this notebook:** Create plots & visualizations using Matplotlib & Seaborn

## What are Matplotlib and Seaborn?

[**Seaborn**](https://seaborn.pydata.org/) (`sns`) creates common charts with short commands, while [**Matplotlib**](https://matplotlib.org/) helps customize titles, labels, and other details. Seaborn is built on Matplotlib, so we can use them together.

### Reading Seaborn code

For example, `sns.scatterplot(data=qbs, x="attempts", y="passing_yards")` creates a scatterplot:
- `data=qbs` selects the pandas DataFrame.
- `x="attempts"` and `y="passing_yards"` select the columns for each axis.
- Optional arguments such as `hue=` (color by a variable), `size=` (scale marker size), and `alpha=` (transparency) add information or change appearance.

These are **arguments** passed to the function. The same `data=`, `x=`, and `y=` pattern works for many Seaborn charts. Use `plt.title()`, `plt.xlabel()`, and `plt.ylabel()` to label your chart, then `plt.show()` to display it.

## Setup
We will be creating data visualizations using NFL data. Instead of downloading a CSV dataset, we will use another Python library called *nflreadpy*, which provides easy access to NFL data. Importing a library allows us to load directly through Python and reduce the number of files we need. If you are interested in finding other sports data libraries, check out our [toolkit](https://github.com/Cornell-Sports-Analytics/toolkit) repo.

Since nflreadpy is not preinstalled in Google Colab, we will need to manually install it using the following command.

*Note: Since nflreadpy does not allow for use with pandas by default, we will also need to install pyarrow. Don't worry, we won't use it directly.*

In [ ]:
%pip install nflreadpy
%pip install pyarrow

With nflreadpy now installed, we will import the Python libraries we will be using in our coding evironment.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nflreadpy as nfl

For today's visualizations, we'll use three types of NFL data: season-level, weekly, and play-by-play data. Therefore, we will need to load the NFL data from nflreadpy and convert it to pandas dataframes.

In [ ]:
# Load 2025 player statistics
players = nfl.load_player_stats(
    [2025], summary_level="reg"
).to_pandas()

# Load weekly player statistics
weekly = nfl.load_player_stats(
    [2025], summary_level="week"
).to_pandas()

# Load play-by-play data
pbp = nfl.load_pbp([2025]).to_pandas()

If you're wondering how I knew which commands to use to load the player statistics, I encourage you to check out the [nflreadpy](https://nflreadpy.nflverse.com/) documentation. Documentation is an important resource when coding because programming isn't about memorizing every command. Instead, it's about knowing how to find and use the information you need. Whenever you're learning a new library, its documentation is usually the best place to start.

## Basic Visualizations

We will go over how to use Seaborn to create 5 basic visualizations: bar charts, scatterplots, line graphs, histograms, & boxplots. Additionally, we will use Matplotlib to further customize our visualizations.

### Bar chart

**Question:** Which running backs rushed for the most yards in the 2025 season?

A **bar chart** compares values across categories. `sns.barplot()` draws the bars; `x=` sets rushing yards and `y=` sets player names. First, we filter for RBs and use `.nlargest(10, "rushing_yards")` to select the top 10.

In [ ]:
# Find the top 10 running backs by rushing yards
top_rushers = players[players["position"] == "RB"]
top_rushers = top_rushers.nlargest(10, "rushing_yards")

# Bar chart
sns.barplot(data=top_rushers, x="rushing_yards", y="player_display_name")
plt.title("Top 10 Running Backs — 2025")
plt.xlabel("Rushing Yards")
plt.ylabel("Player")
plt.show()

### Scatterplot

**Question:** Do quarterbacks who attempt more passes usually throw for more yards?

A **scatterplot** shows how two numerical variables relate. Each point is one quarterback. The filters below keep QBs with at least 100 attempts so very low-volume passers do not dominate the display.

In [ ]:
# Keep quarterbacks with at least 100 attempts
qbs = players[players["position"] == "QB"]
qbs = qbs[qbs["attempts"] >= 100]

# Scatter plot
sns.scatterplot(data=qbs, x="attempts", y="passing_yards")
plt.title("QB Passing Attempts vs. Passing Yards")
plt.xlabel("Passing Attempts")
plt.ylabel("Passing Yards")
plt.show()

### Line Plot

**Question:** How did the top receiver’s yardage change throughout the season?

A **line plot** tracks values over an ordered variable, such as week. We find the season's leading WR, match their `player_id` to the weekly data, and use `.sort_values("week")` to put the observations in order.

In [ ]:
# Find the receiver with the most 2025 receiving yards
receivers = players[players["position"] == "WR"]
top_receiver = receivers.nlargest(1, "receiving_yards").iloc[0]

# Get that player's weekly stats
receiver_weeks = weekly[weekly["player_id"] == top_receiver["player_id"]]
receiver_weeks = receiver_weeks.sort_values("week")

# Line chart
sns.lineplot(data=receiver_weeks, x="week", y="receiving_yards", marker="o")
plt.title(top_receiver["player_display_name"] + "'s Weekly Receiving Yards")
plt.xlabel("Week")
plt.ylabel("Receiving Yards")
plt.show()

**Note:** `.nlargest(1, "receiving_yards")` returns the top row as a one-row DataFrame; `.iloc[0]` retrieves that row so we can access its player ID and name.

### Histogram

**Question:** How many yards does an NFL rushing attempt typically gain?

A **histogram** groups numerical values into intervals (bins) and shows how often each interval occurs. We filter for recorded rush attempts, exclude two-point attempts, and use `sns.histplot()` to display the distribution.

In [ ]:
# Keep rushing plays with recorded yardage
rush_plays = pbp[pbp["play_type"] == "run"]
rush_plays = rush_plays[rush_plays["rush_attempt"] == 1]
rush_plays = rush_plays[rush_plays["two_point_attempt"] != 1]
rush_plays = rush_plays.dropna(subset=["yards_gained"])

# Histogram of yards gained on rushing attempts
sns.histplot(data=rush_plays, x="yards_gained", bins=30)
plt.title("Yards Gained on NFL Rushing Attempts (2025)")
plt.xlabel("Yards Gained")
plt.ylabel("Number of Attempts")
plt.show()

### Box Plot

**Question:** Which of the two leading running backs has more consistent rushing gains?

A **boxplot** summarizes a distribution using its median, quartiles, and possible outliers. `.head(2)` selects the first two rows of our already sorted `top_rushers` table. `.isin(...)` keeps carries by either player, and `.map(names)` turns player IDs into readable names.

In [ ]:
# Find the two leading running backs
best_two = top_rushers.head(2)

# Get their individual carries
rb_runs = rush_plays[rush_plays["rusher_player_id"].isin(best_two["player_id"])].copy()

# Add readable player names
names = dict(zip(best_two["player_id"], best_two["player_display_name"]))
rb_runs["player"] = rb_runs["rusher_player_id"].map(names)

# Box plot
sns.boxplot(data=rb_runs, x="player", y="yards_gained")
plt.title("Rushing Yards per Carry — Two Leading RBs")
plt.xlabel("Running Back")
plt.ylabel("Yards Gained on Each Carry")
plt.ylim(-8,28)
plt.show()

### Wrap-up
| Chart | Best for | Seaborn function |
|---|---|---|
| Bar | Comparing categories | `sns.barplot()` |
| Scatter | Finding relationships | `sns.scatterplot()` |
| Line | Tracking changes by week | `sns.lineplot()` |
| Histogram | Showing a distribution | `sns.histplot()` |
| Box | Comparing distributions | `sns.boxplot()` |

### Exercise 1: Basic Visualizations

**Create a new bar chart from scratch:** Which **five wide receivers** scored the most receiving touchdowns in 2025?

1. Use `players` to select wide receivers (`position == "WR"`).
2. Find the top five by `receiving_tds`.
3. Create a bar chart with player names and receiving touchdowns.
4. Add a title and axis labels. What do you notice?

*Hint:* Look back at the running back bar chart for the filtering, `.nlargest()`, and `sns.barplot()` steps. Try writing your code in the cell block below.


In [ ]:
# INSERT CODE HERE

## Advanced Visualizations

Here we'll explore three more advanced visualizations: bubble charts, heatmaps, and football field plots. These use extra visual encodings or data preparation to answer more detailed questions.

### Bubble Chart

**Question:** How do wide receivers' targets, receiving yards, and receiving touchdowns relate?

A **bubble chart** is a scatterplot with an extra variable represented by bubble size. Here `x=` and `y=` show targets and yards; `size=` and `hue=` use touchdowns for marker size and color. `sizes=(20, 400)` sets the range of bubble areas, and `alpha=0.7` adds transparency.

In [ ]:
# Select wide receivers
wrs = players[players["position"] == "WR"]

# Create bubble chart
sns.scatterplot(
    data=wrs,
    x="targets",
    y="receiving_yards",
    size="receiving_tds",
    hue="receiving_tds",
    sizes=(20, 400),
    alpha=0.7
)

plt.title("NFL Wide Receiver Production (2025)")
plt.xlabel("Targets")
plt.ylabel("Receiving Yards")
plt.show()

### Heatmap

**Question:** Which running back statistics tend to move together?

A **correlation heatmap** shows pairwise relationships between numerical columns. We select RB stats with `rbs[[...]]`, use `.corr()` to calculate their correlations (from −1 to 1), then pass that table to `sns.heatmap()`. `annot=True` prints values in cells, `cmap="coolwarm"` chooses the colors, and `vmin=-1`, `vmax=1`, `center=0` fix the scale around zero.

In [ ]:
# Select running backs
rbs = players[players["position"] == "RB"]

# Choose statistics to compare
stats = rbs[[
    "carries",
    "rushing_yards",
    "rushing_tds",
    "targets",
    "receiving_yards",
    "receiving_tds"
]]

# Calculate correlations
correlations = stats.corr()

# Create correlation heatmap
sns.heatmap(
    correlations,
    annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0
)

plt.title("Running Back Statistics Correlation (2025)")
plt.show()

### Football Field Plot

**Question:** From which yard lines do rushing touchdowns begin?

We filter to rushing touchdown plays and use `.dropna(subset=["yardline_100"])` to remove plays without a recorded yard line. `field_x` converts yards from the opponent's end zone into a position on a 120-yard drawing (including end zones). Matplotlib's `plt.axvspan()` colors the field and `plt.axvline()` draws yard lines; Seaborn plots the points.

**Note:** The data does **not** contain lateral (side-to-side) positions. `field_y` assigns a random display position to prevent all points overlapping; only the **lengthwise** starting yard line comes from the data.

In [ ]:
# Find rushing touchdowns
run_tds = pbp[
    (pbp["play_type"] == "run") &
    (pbp["rush_touchdown"] == 1)
].dropna(subset=["yardline_100"]).copy()

# Calculate field positions
run_tds["field_x"] = 110 - run_tds["yardline_100"]
run_tds["field_y"] = np.random.choice(
    [23.6, 26.65, 29.7],
    size=len(run_tds)) # randomly generate y-coord since nflreadpy doesn't include it

# Create football field
plt.figure(figsize=(12, 5))
plt.axvspan(0, 120, color="forestgreen")

# Draw yard lines
for x in range(10, 111, 10):
    plt.axvline(x, color="white", alpha=0.5)

# Plot rushing touchdown locations
sns.scatterplot(
    data=run_tds,
    x="field_x",
    y="field_y",
    color="gold",
    s=70
)

plt.xlim(0, 120)
plt.ylim(0, 53.3)
plt.title("Rushing Touchdown Starting Positions (2025)")
plt.xlabel("Field Position")
plt.yticks([])
plt.show()

### Wrap-up

| Chart | Best for | Function |
|---|---|---|
| Bubble | Comparing three numerical variables | `sns.scatterplot()` |
| Heatmap | Finding correlations between variables | `sns.heatmap()` + `.corr()` |
| Football Field | Visualizing play locations on a field | `plt.axvspan()`, `plt.axvline()`, `sns.scatterplot()` |

### Exercise 2: Advanced Visualizations

**Create a new bubble chart from scratch:** How do quarterbacks' **passing attempts**, **passing yards**, and **passing touchdowns** relate?

1. Start from `players` and select quarterbacks (`position == "QB"`) with at least 100 passing attempts.
2. Build a bubble chart with `attempts` on the x-axis, `passing_yards` on the y-axis, and bubble **size** representing `passing_tds`.
3. Add a title and axis labels.
4. Do quarterbacks with similar attempts always have similar touchdown totals?

*Hint:* Use `sns.scatterplot()` and the `size=` argument from the bubble chart example. Try writing your code in the cell block below.


In [ ]:
# INSERT CODE HERE